# LINet Training on SUN RGB-D - Google Colab

**Hyperparameter tuning with Ray Tune using a locked 80/20 train/val split**

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100
- [ ] **Upload dataset to Drive:** `MyDrive/datasets/sunrgbd_19_traintest.tar.gz`
- [ ] **(Optional)** Upload pretrained weights for transfer learning


## 1. Environment Setup & GPU Verification

In [1]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

# Check PyTorch and CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    # Check if it's A100
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\n✅ A100 GPU detected - PERFECT for training!")
    elif 'V100' in gpu_name:
        print("\n✅ V100 GPU detected - Good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\n⚠️  T4 GPU detected - Will be slower, consider upgrading to A100")
    else:
        print(f"\n⚠️  GPU: {gpu_name} - Consider using A100 for best performance")
else:
    print("\n❌ NO GPU DETECTED!")
    print("Please enable GPU: Runtime → Change runtime type → Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

GPU VERIFICATION
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU Device: NVIDIA A100-SXM4-40GB
GPU Memory: 39.49 GB

✅ A100 GPU detected - PERFECT for training!



In [2]:
# Detailed GPU info
!nvidia-smi

Fri Apr  3 16:07:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             45W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [3]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

Mounted at /content/drive

✅ Google Drive mounted successfully!

Drive contents:
total 3117908
-rw------- 1 root root        176 Sep 21  2019 06-lab2.gdoc
-rw------- 1 root root      21621 Sep 30  2024 113-1363667-3121001@USSR24093000064918@pre-paid.png
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (1).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (2).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (3).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final.gdoc
-rw------- 1 root root        176 Jul 11  2025 2025_Gabriel_Clinger_Contractor Agreement_BASE copy.gdoc
-rw------- 1 root root      32204 Apr 18  2022 2900 On First- Welcome Home Next Steps.docx
-rw------- 1 root root       8822 Jun 24  2017 A6.docx
-rw------- 1 root root      22204 Jan 21  2023 activity (1).xlsx
-rw------- 1 root root      22161 Jan 21  2023 activity (2).xlsx
-rw------- 1 root root        176 Jan 21  2023 activity.gsheet
-rw------- 

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [4]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"  # UPDATE THIS
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"  # Local copy for fast I/O

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

# Ensure we're in a valid directory
os.chdir('/content')
print(f"Starting in: {os.getcwd()}")

# Check if repo already exists (same session, rerunning cell)
if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"\n📁 Repo already exists: {LOCAL_REPO_PATH}")
    print(f"🔄 Pulling latest changes...")

    os.chdir(LOCAL_REPO_PATH)
    !git pull
    print("✅ Repo updated")

# Clone from GitHub (first run)
else:
    # Remove old incomplete copy if exists
    if Path(LOCAL_REPO_PATH).exists():
        print(f"\n🗑️  Removing incomplete repo copy...")
        !rm -rf {LOCAL_REPO_PATH}

    print(f"\n🔄 Cloning from GitHub...")
    print(f"   Repo: {GITHUB_REPO}")
    print(f"   Destination: {LOCAL_REPO_PATH}")

    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}

    # Verify clone succeeded
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository to {LOCAL_REPO_PATH}")

    print("✅ Repo cloned successfully")
    os.chdir(LOCAL_REPO_PATH)

# Verify repo structure
print(f"\n📂 Repository structure:")
!ls -la {LOCAL_REPO_PATH}

print(f"\n✅ Working directory: {os.getcwd()}")

REPOSITORY SETUP
Starting in: /content

🔄 Cloning from GitHub...
   Repo: https://github.com/clingergab/Multi-Stream-Neural-Networks.git
   Destination: /content/Multi-Stream-Neural-Networks
Cloning into '/content/Multi-Stream-Neural-Networks'...
remote: Enumerating objects: 3299, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 3299 (delta 82), reused 75 (delta 75), pack-reused 3208 (from 2)
Receiving objects: 100% (3299/3299), 122.37 MiB | 36.81 MiB/s, done.
Resolving deltas: 100% (2085/2085), done.
Encountered 49 file(s) that should have been pointers, but weren't:
	tests/augmentation_comparison.png
	tests/augmentation_test.png
	tests/balanced_augmentation_test.png
	tests/balanced_samples_comparison.png
	tests/dataset_orthogonal_loading.png
	tests/decaying_restarts_eta_min_bug.png
	tests/easing_formula_analysis.png
	tests/easing_schedulers_comparison.png
	tests/global_vs_local_comparison.png
	tests/linear_scale_compar

## 4. Install Dependencies

In [5]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] optuna kornia

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import ray
import kornia

print("✅ All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   ray: {ray.__version__}")
print(f"   kornia: {kornia.__version__}")


Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 35.1 MB/s eta 0:00:00:00:0100:01
✅ All dependencies installed!
   h5py: 3.16.0
   matplotlib: 3.10.0
   ray: 2.54.1
   kornia: 0.8.2


## 5. Copy SUN RGB-D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** SUN RGB-D 19-category preprocessed dataset with RGB + Depth


In [6]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sunrgbd_19_traintest"

print("=" * 60)
print("SUN RGB-D 19-CATEGORY DATASET SETUP")
print("=" * 60)

if Path(LOCAL_DATASET_PATH).exists():
    print(f"Already on local disk: {LOCAL_DATASET_PATH}")
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"   Train samples: {train_count}")
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found on Drive: {DRIVE_DATASET_TAR}")
    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"
    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"Extracted. Train samples: {train_count}")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_TAR}")

print(f"\nDataset ready at: {LOCAL_DATASET_PATH}")


SUN RGB-D 19-CATEGORY DATASET SETUP
Found on Drive: /content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz
          1.66G 100%   81.49MB/s    0:00:19 (xfr#1, to-chk=0/1)

Extracting...
Extracted. Train samples: 0

Dataset ready at: /dev/shm/sunrgbd_19_traintest


## 6. Setup Python Path & Import LINet


In [7]:
import sys
import os

modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project structure:")
!ls -la {project_root}/src/models/

print("\nImporting LiNet and dataloaders...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders, SUNRGBDDataset
from src.training.augmentation_config import AugmentationConfig

from ray import train, tune
from ray.tune.schedulers import ASHAScheduler

print("All imports successful!")


Project structure:
total 48
drwxr-xr-x 11 root root 4096 Apr  3 16:07 .
drwxr-xr-x  7 root root 4096 Apr  3 16:07 ..
drwxr-xr-x  2 root root 4096 Apr  3 16:07 abstracts
drwxr-xr-x  2 root root 4096 Apr  3 16:07 common
drwxr-xr-x  2 root root 4096 Apr  3 16:07 core
drwxr-xr-x  2 root root 4096 Apr  3 16:07 direct_mixing_activation
drwxr-xr-x  2 root root 4096 Apr  3 16:07 direct_mixing_bn
drwxr-xr-x  2 root root 4096 Apr  3 16:07 direct_mixing_conv
-rw-r--r--  1 root root 1076 Apr  3 16:07 __init__.py
drwxr-xr-x  4 root root 4096 Apr  3 16:07 linear_integration
drwxr-xr-x  2 root root 4096 Apr  3 16:07 multi_channel
drwxr-xr-x  2 root root 4096 Apr  3 16:07 utils

Importing LiNet and dataloaders...
All imports successful!


## 8b. Hyperparameter Tuning with Ray Tune

- **Parallel Trials:** Run multiple configurations simultaneously
- **Locked 80/20 Split:** Deterministic stratified train/val split
- **ASHA Scheduler:** Early-stop unpromising trials
- **Resumable:** Experiment state saved to Google Drive, survives Colab restarts
- **Optional Pretrained Weights:** Load Omni backbone for transfer learning


In [8]:
import os
import time

# 1. Define Paths explicitly
mps_pipe_dir = "/tmp/nvidia-mps"
mps_log_dir = "/tmp/nvidia-log"

# 2. Create the directories (CRITICAL: Daemon fails if log dir doesn't exist)
os.makedirs(mps_pipe_dir, exist_ok=True)
os.makedirs(mps_log_dir, exist_ok=True)

# 3. Set Environment Variables for the current Python process
os.environ["CUDA_MPS_PIPE_DIRECTORY"] = mps_pipe_dir
os.environ["CUDA_MPS_LOG_DIRECTORY"] = mps_log_dir
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# 4. Configure GPU and Start Daemon using the SAME environment variables
# We use f-strings to pass the python variables into the shell command
print("Setting GPU to Exclusive Process Mode...")
!nvidia-smi -i 0 -c EXCLUSIVE_PROCESS

print("Starting MPS Daemon...")
# We explicitly pass the env vars to the shell command
!export CUDA_MPS_PIPE_DIRECTORY={mps_pipe_dir} && \
 export CUDA_MPS_LOG_DIRECTORY={mps_log_dir} && \
 nvidia-cuda-mps-control -d

# 5. Verify it is running
print("Verifying Daemon Status...")
time.sleep(1) # Give it a second to start
!ps -ef | grep mps

# Check if the pipe file actually exists
if os.path.exists(os.path.join(mps_pipe_dir, "control")):
    print("✅ MPS Control Pipe found. Setup success.")
else:
    print("❌ MPS Control Pipe NOT found. Check /tmp/nvidia-log for errors.")
    # Optional: Print logs if it failed
    !cat {mps_log_dir}/control.log

Setting GPU to Exclusive Process Mode...
Set compute mode to EXCLUSIVE_PROCESS for GPU 00000000:00:04.0.
All done.
Starting MPS Daemon...
Verifying Daemon Status...
root        3846       1  0 16:09 ?        00:00:00 nvidia-cuda-mps-control -d
root        3852    3044  0 16:09 ?        00:00:00 /bin/bash -c ps -ef | grep mps
root        3854    3852  0 16:09 ?        00:00:00 grep mps
✅ MPS Control Pipe found. Setup success.


In [ ]:
import random
import numpy as np

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch
from optuna.samplers import TPESampler
from ray.tune.schedulers import MedianStoppingRule
import torch
from collections import Counter
from sklearn.model_selection import train_test_split

from src.models.linear_integration.li_net3 import li_resnet18
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.data_utils.sunrgbd_dataset import SUNRGBDDataset
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed
from src.data_utils.sunrgbd_dataset import _load_norm_stats
from src.models.common.model_helpers import load_pretrained_backbone


class TrialTerminated(Exception):
    """Raised when a trial should be terminated early."""
    pass


class RayTuneReporter:
    """Callback for reporting metrics to Ray Tune during training."""

    def __init__(self):
        self.best_accuracy = 0.0
        self.best_loss = float('inf')
        self.best_train_acc = 0.0
        self.best_val_mca = 0.0
        self.best_train_mca = 0.0
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs):
        """Report current AND best metrics to Ray Tune."""
        if logs['val_accuracy'] > self.best_accuracy:
            self.best_accuracy = logs['val_accuracy']
            if logs['train_accuracy'] > self.best_train_acc:
                self.best_train_acc = logs['train_accuracy']

        if logs['val_loss'] < self.best_loss:
            self.best_loss = logs['val_loss']

        val_mca = logs.get('val_mca', 0.0)
        train_mca = logs.get('train_mca', 0.0)
        if val_mca > self.best_val_mca:
            self.best_val_mca = val_mca
            self.best_epoch = epoch
            if train_mca > self.best_train_mca:
                self.best_train_mca = train_mca

        gap = self.best_train_mca - self.best_val_mca
        composite = self.best_val_mca - 5 * (gap**3)

        tune.report({
            "accuracy": logs['val_accuracy'],
            "loss": logs['val_loss'],
            "best_accuracy": self.best_accuracy,
            "best_loss": self.best_loss,
            "train_loss": logs['train_loss'],
            "train_accuracy": logs['train_accuracy'],
            "best_train_acc": self.best_train_acc,
            "val_mca": val_mca,
            "train_mca": train_mca,
            "best_val_mca": self.best_val_mca,
            "best_train_mca": self.best_train_mca,
            "composite": composite,
            "best_epoch": self.best_epoch,
        })


def train_linet_tune(
    config,
    data_root=None,
    norm_stats=None,
    pretrained_weights_path=None,
    freeze_backbone_epochs=0,
    freeze_backbone_lr=1e-3,
    seed=42,
):
    """
    Trainable function for Ray Tune — locked 80/20 stratified split.

    Args:
        config: Ray Tune configuration dict with hyperparameters
        data_root: Path to dataset root (with train/ directory)
        norm_stats: Normalization statistics dict
        pretrained_weights_path: Path to pretrained checkpoint (or None)
        seed: Random seed for reproducible trials
    """
    set_seed(seed, deterministic=False)
    g = torch.Generator().manual_seed(seed)

    # Per-trial augmentation config
    aug_config = AugmentationConfig(
        rgb_aug_prob=config.get("rgb_aug_prob"),
        rgb_aug_mag=config.get("rgb_aug_mag"),
        depth_aug_prob=config.get("depth_aug_prob"),
        depth_aug_mag=config.get("depth_aug_mag"),
    )

    # Two dataset instances from the same train/ directory:
    # 1) train_dataset: augmentation ON (split='train')
    # 2) val_dataset:   augmentation OFF (split overridden to 'val')
    train_dataset = SUNRGBDDataset(
        data_root=data_root,
        split='train',
        normalize=False,  # GPU will normalize after augmentation
        **aug_config.to_dict(),
    )
    val_dataset = SUNRGBDDataset(
        data_root=data_root,
        split='train',
        normalize=False,
    )
    val_dataset.split = 'val'  # Disable augmentation in __getitem__

    # Locked 80/20 stratified split (deterministic — same split every trial)
    all_labels = train_dataset.labels
    train_indices, val_indices = train_test_split(
        list(range(len(all_labels))),
        test_size=0.2,
        random_state=seed,
        stratify=all_labels,
    )

    train_subset = torch.utils.data.Subset(train_dataset, train_indices)
    val_subset = torch.utils.data.Subset(val_dataset, val_indices)

    # Stratified sampling for training
    subset_labels = [all_labels[i] for i in train_indices]
    label_counts = Counter(subset_labels)
    num_samples = len(subset_labels)
    class_weights = {label: num_samples / count for label, count in label_counts.items()}
    sample_weights = torch.tensor(
        [class_weights[label] for label in subset_labels], dtype=torch.float32
    )

    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=num_samples,
        replacement=True,
        generator=g,
    )

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = torch.utils.data.DataLoader(
        train_subset,
        batch_size=64,
        shuffle=False,
        sampler=train_sampler,
        num_workers=1,
        prefetch_factor=2,
        persistent_workers=True,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )
    val_loader = torch.utils.data.DataLoader(
        val_subset,
        batch_size=64,
        shuffle=False,
        num_workers=1,
        prefetch_factor=2,
        persistent_workers=False,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )

    # Create Model
    model = li_resnet18(
        num_classes=19,
        stream_input_channels=[3, 1],
        dropout_p=config["dropout_p"],
        width_multiplier=0.75,
        device="cuda",
        use_amp=True,
    )

    # Load pretrained backbone weights (if provided)
    if pretrained_weights_path is not None:
        load_pretrained_backbone(model, pretrained_weights_path, verbose=False)

        # Optional backbone freeze warmup: train only classifier head with static LR
        if freeze_backbone_epochs > 0:
            for param in model.parameters():
                param.requires_grad = False
            for param in model.fc.parameters():
                param.requires_grad = True

            fc_optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=freeze_backbone_lr,
            )
            model.compile(
                optimizer=fc_optimizer,
                scheduler=None,
                loss='cross_entropy',
                label_smoothing=config["label_smoothing"],
                gpu_augmentation=True,
                norm_stats=norm_stats,
                **aug_config.to_dict(),
            )
            model.fit(
                train_loader=train_loader,
                val_loader=val_loader,
                epochs=freeze_backbone_epochs,
                verbose=False,
                modality_dropout=False,
            )

            # Unfreeze all parameters for the full trial
            for param in model.parameters():
                param.requires_grad = True

    # Create Optimizer
    optimizer = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=[config["lr"], config["lr"]],
        stream_weight_decays=[config["wd"], config["wd"]],
        shared_lr=config["lr"],
        integration_weight_decay=config["wd"],
        stem_lr_multiplier=config["stem_lr_multiplier"],
    )

    # Create Scheduler
    warmup_epochs = 5
    scheduler = setup_scheduler(
        optimizer,
        scheduler_type='cosine',
        # eta_min: 6 groups when stem split, 4 when not
        eta_min=(
            [config['eta_min'] * config['stem_lr_multiplier'],
             config['eta_min'] * config['stem_lr_multiplier'],
             config['eta_min'], config['eta_min'],
             config['eta_min'], config['eta_min']]
            if config['stem_lr_multiplier'] != 1.0 else
            [config['eta_min']] * 4
        ),
        t_max=110,
        train_loader_len=len(train_loader),
        warmup_epochs=warmup_epochs,
        warmup_start_factor=0.2,
    )

    # Compile
    model.compile(
        optimizer=optimizer,
        scheduler=scheduler,
        loss='cross_entropy',
        label_smoothing=config["label_smoothing"],
        gpu_augmentation=True,
        norm_stats=norm_stats,
        **aug_config.to_dict(),
    )

    # Train
    try:
        model.fit(
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=115,
            early_stopping=True,
            patience=15,
            monitor='val_mca',
            grad_clip_norm=config["grad_clip_norm"],
            modality_dropout=True,
            modality_dropout_start=0,
            modality_dropout_ramp=20,
            modality_dropout_rate=0.4,
            callbacks=[RayTuneReporter()],
            verbose=False,
        )
    except TrialTerminated as e:
        print(f"\n{e}")


In [10]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# Ray Tune saves ALL experiment state to DRIVE_STORAGE_PATH via storage_path.
# When Colab dies, re-run the notebook — Tuner.restore() picks up where it
# left off. Completed trials preserved, interrupted trials restart.
# =============================================================================

import hashlib
import json as json_module
import os
import pandas as pd
from pathlib import Path

# --- Ray Tune persistent storage on Google Drive ---
DRIVE_STORAGE_PATH = "/content/drive/MyDrive/ray_tune_experiments"
LOCAL_STORAGE_PATH = "/content/ray_results"
EXPERIMENT_NAME = "sun_rgbd_hpo_opt_mca_high_MD"

SEED = 42
NUM_SAMPLES = 200  # Total trials to run across all sessions

# --- Pretrained Weights (Optional) ---
# Set LOAD_WEIGHTS = True to initialize every trial from pretrained backbone
# weights (e.g. from OmniObject3D pretraining). The fc head is skipped
# automatically if num_classes differs.
LOAD_WEIGHTS = False
PRETRAINED_WEIGHTS_PATH = "/content/drive/MyDrive/linet_checkpoints/omni_best/final_model.pt"

# --- Backbone Freeze Warmup (Optional, only used when LOAD_WEIGHTS = True) ---
# Freeze pretrained backbone for N epochs, training only the classifier head
# with a static LR before the full HPO trial begins.
FREEZE_BACKBONE_EPOCHS = 0    # Set > 0 to enable (e.g. 3-5 epochs)
FREEZE_BACKBONE_LR = 1e-3     # Static LR for classifier warmup


Path(DRIVE_STORAGE_PATH).mkdir(parents=True, exist_ok=True)
Path(LOCAL_STORAGE_PATH).mkdir(parents=True, exist_ok=True)

experiment_path = os.path.join(DRIVE_STORAGE_PATH, EXPERIMENT_NAME)
local_experiment_path = os.path.join(LOCAL_STORAGE_PATH, EXPERIMENT_NAME)
RESUME_EXISTING = os.path.exists(experiment_path)

if RESUME_EXISTING:
    # Validate experiment dir has actual content
    _exp_files = os.listdir(experiment_path) if os.path.isdir(experiment_path) else []
    if len(_exp_files) == 0:
        print(f"  WARNING: {experiment_path} exists but is empty \u2014 starting fresh")
        RESUME_EXISTING = False

print(f"Drive storage: {DRIVE_STORAGE_PATH}")
print(f"Local storage: {LOCAL_STORAGE_PATH}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Resume existing: {RESUME_EXISTING}")
print(f"Total trials: {NUM_SAMPLES}")
print(f"Load pretrained weights: {'ENABLED' if LOAD_WEIGHTS else 'DISABLED'}")
if LOAD_WEIGHTS:
    print(f"   Weights: {PRETRAINED_WEIGHTS_PATH}")
    print(f"   Freeze warmup: {FREEZE_BACKBONE_EPOCHS} epochs (LR={FREEZE_BACKBONE_LR})" if FREEZE_BACKBONE_EPOCHS > 0 else "   Freeze warmup: DISABLED")
if RESUME_EXISTING:
    print(f"\n  Previous experiment found at {experiment_path}")
    print(f"  Copying Drive -> local, then Tuner.restore() from local.")
    # Copy experiment state from Drive to local before Tuner.restore
    import subprocess as _sp_cfg
    os.makedirs(local_experiment_path, exist_ok=True)
    _result = _sp_cfg.run(
        ["rsync", "-a", experiment_path + "/", local_experiment_path + "/"],
        capture_output=True, text=True,
    )
    if _result.returncode == 0:
        print(f"  Restored to {local_experiment_path}")
    else:
        raise RuntimeError(f"Restore failed: {_result.stderr[:300]}")


Drive storage: /content/drive/MyDrive/ray_tune_experiments
Local storage: /content/ray_results
Experiment: sun_rgbd_hpo_opt_mca_high_MD
Resume existing: False
Total trials: 200
Load pretrained weights: DISABLED


In [ ]:
# Initialize Ray
import shutil
import subprocess
import time as _time

from ray.tune import CLIReporter
from ray.tune import Callback as TuneCallback

os.environ["RAY_AIR_NEW_OUTPUT"] = "0"  # must be set BEFORE ray.init()

class DriveSyncCallback(TuneCallback):
    """Periodically rsyncs local Ray Tune experiment state to Google Drive.

    Ray Tune writes to LOCAL_STORAGE_PATH (fast local disk).  This callback
    rsyncs local -> Drive incrementally (only changed files) so that state
    survives Colab session death without blocking the Ray driver.
    """

    def __init__(self, local_storage_path, drive_storage_path, experiment_name,
                 sync_interval_seconds=300):
        self._local_path = os.path.join(local_storage_path, experiment_name)
        self._drive_path = os.path.join(drive_storage_path, experiment_name)
        self._sync_interval = sync_interval_seconds
        self._last_sync = 0.0

    def _sync(self, reason=""):
        if not os.path.isdir(self._local_path):
            return
        try:
            os.makedirs(self._drive_path, exist_ok=True)
            result = subprocess.run(
                ["rsync", "-a",
                 self._local_path + "/",
                 self._drive_path + "/"],
                capture_output=True, text=True, timeout=120,
            )
            if result.returncode == 0:
                self._last_sync = _time.time()
                print(f"[DriveSyncCallback] synced to Drive ({reason})")
            else:
                print(f"[DriveSyncCallback] WARNING: rsync failed: {result.stderr[:200]}")
        except subprocess.TimeoutExpired:
            print(f"[DriveSyncCallback] WARNING: rsync timed out (120s)")
        except Exception as e:
            print(f"[DriveSyncCallback] WARNING: sync failed: {e}")

    def on_trial_result(self, iteration, trials, trial, result, **info):
        if _time.time() - self._last_sync >= self._sync_interval:
            self._sync(reason=f"periodic, iter={result.get('training_iteration', '?')}")

    def on_trial_complete(self, iteration, trials, trial, **info):
        if _time.time() - self._last_sync >= 60:
            self._sync(reason="trial complete")

    def on_experiment_end(self, trials, **info):
        self._sync(reason="experiment end")


class BestTrialReporter(TuneCallback):
    """Periodically prints the best trial's config and metrics."""

    def __init__(self, metric="best_val_mca", mode="max", every_n_results=20):
        self._metric = metric
        self._mode = mode
        self._every_n = every_n_results
        self._result_count = 0
        self._best_value = float('-inf') if mode == "max" else float('inf')
        self._best_config = None

    def on_trial_result(self, iteration, trials, trial, result, **info):
        self._result_count += 1
        val = result.get(self._metric, None)
        if val is None:
            return
        improved = (val > self._best_value) if self._mode == "max" else (val < self._best_value)
        if improved:
            self._best_value = val
            self._best_config = trial.config.copy()

        if self._result_count % self._every_n == 0 and self._best_config is not None:
            self._print_best(result)

    def _print_best(self, latest_result):
        print(f"\n{'─'*60}")
        print(f"  ★ Best {self._metric}: {self._best_value*100:.2f}% "
              f"(after {self._result_count} results)")
        for k, v in self._best_config.items():
            if isinstance(v, float):
                print(f"    {k}: {v:.2e}" if abs(v) < 0.01 else f"    {k}: {v:.4f}")
            else:
                print(f"    {k}: {v}")
        print(f"{'─'*60}")


ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    runtime_env={
        "env_vars": {
            "CUDA_MPS_PIPE_DIRECTORY": "/tmp/nvidia-mps",
            "CUDA_MPS_LOG_DIRECTORY": "/tmp/nvidia-log",
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            "CUDA_VISIBLE_DEVICES": "0",
        }
    }
)

norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)

print(f"Dataset: {LOCAL_DATASET_PATH}")


# Define trainable (same for both new and restored runs)
trainable = tune.with_resources(
    tune.with_parameters(
        train_linet_tune,
        data_root=LOCAL_DATASET_PATH,
        norm_stats=norm_stats,
        seed=SEED,
        pretrained_weights_path=PRETRAINED_WEIGHTS_PATH if LOAD_WEIGHTS else None,
        freeze_backbone_epochs=FREEZE_BACKBONE_EPOCHS if LOAD_WEIGHTS else 0,
        freeze_backbone_lr=FREEZE_BACKBONE_LR,
    ),
    resources={"cpu": 1, "gpu": 0.1},
)


# Callback to force-sync experiment state to Drive
drive_sync_cb = DriveSyncCallback(LOCAL_STORAGE_PATH, DRIVE_STORAGE_PATH, EXPERIMENT_NAME)


if RESUME_EXISTING:
    # =========================================================
    # RESUME: Restore previous experiment from Google Drive
    # =========================================================
    print("\n" + "=" * 60)
    print("RESUMING EXPERIMENT FROM GOOGLE DRIVE")
    print("=" * 60)

    tuner = tune.Tuner.restore(
        path=local_experiment_path,
        trainable=trainable,
        resume_unfinished=True,
        resume_errored=True,
    )

else:
    # =========================================================
    # NEW: Create fresh experiment
    # =========================================================
    print("\n" + "=" * 60)
    print("STARTING NEW EXPERIMENT")
    print("=" * 60)

    # Search space (SUN RGB-D tuned ranges)
    search_space = {
        # Learning rates
        "lr": tune.uniform(9.0e-5, 3e-4),
        "wd": tune.loguniform(7.0e-5, 2e-4),

        # Scheduler eta_min
        "eta_min": tune.loguniform(5.0e-7, 5.0e-6),

        # batch size
        # "batch_size": tune.choice([64]),

        # Regularization
        "dropout_p": tune.quniform(0.25, 0.4, 0.01),
        "label_smoothing": tune.quniform(0.12, 0.14, 0.01),
        "grad_clip_norm": tune.quniform(0.8, 1.2, 0.1),

        # Stem learning rate multiplier (DLR for conv1)
        "stem_lr_multiplier": tune.quniform(2.0, 15.0, 1.0),

        # Augmentation parameters
        "rgb_aug_prob": tune.quniform(0.7, 1.0, 0.01),
        "rgb_aug_mag": tune.quniform(0.7, 1.0, 0.01),
        "depth_aug_prob": tune.quniform(0.7, 1.0, 0.01),
        "depth_aug_mag": tune.quniform(0.7, 1.0, 0.01),

        # Modality dropout
        "modality_dropout_rate": tune.quniform(0.45, 0.50, 0.01),
    }



    reporter = CLIReporter(
        parameter_columns=[
            "lr",
            "wd",
            "eta_min",
            # "t_max", "batch_size",
            "dropout_p",
            "label_smoothing",
            "grad_clip_norm",
            'stem_lr_multiplier',
            "rgb_aug_prob",
            "rgb_aug_mag",
            "depth_aug_prob",
            "depth_aug_mag",
            "modality_dropout_rate",
            # "modality_dropout_start",
            #"modality_dropout_ramp",
        ],
        metric_columns={
            "training_iteration": "iter",
            "best_val_mca": "best_val_mca",
            "best_train_mca": "best_train_mca",
            "best_accuracy": "best_accuracy",
            "composite": "composite",
            "best_epoch": "best_epoch",
        },
        max_report_frequency=30,
        print_intermediate_tables=True,
    )

    optuna_search = OptunaSearch(
        metric="best_val_mca",
        mode="max",
        sampler=TPESampler(n_startup_trials=15),
    )

    asha_scheduler = ASHAScheduler(
        time_attr="training_iteration",
        metric="best_val_mca",
        mode="max",
        max_t=115,
        grace_period=20,
        reduction_factor=2,
    )

    # msr_scheduler = MedianStoppingRule(
    #     time_attr="training_iteration",
    #     metric="best_val_mca",
    #     mode="max",
    #     grace_period=15,
    #     min_samples_required=3,
    # )

    tuner = tune.Tuner(
        trainable,
        param_space=search_space,
        tune_config=tune.TuneConfig(
            scheduler=asha_scheduler,
            search_alg=optuna_search,
            num_samples=NUM_SAMPLES,
            max_concurrent_trials=10,
        ),
        run_config=ray.tune.RunConfig(
            storage_path=LOCAL_STORAGE_PATH,
            name=EXPERIMENT_NAME,
            progress_reporter=reporter,
            verbose=1,
            callbacks=[drive_sync_cb, BestTrialReporter(every_n_results=40)],
        ),
    )


# Run (or resume) tuning
print("\n" + "=" * 60)
print("STARTING HYPERPARAMETER TUNING")
print("=" * 60)

results = tuner.fit()

best_result = results.get_best_result("best_val_mca", "max")

print("\n" + "=" * 60)
print("TUNING COMPLETE")
print("=" * 60)
print(f"Best Trial Config: {best_result.config}")
print(f"Best Trial Val MCA: {best_result.metrics['best_val_mca']:.4f}")
print(f"Best Trial Accuracy: {best_result.metrics['best_accuracy']:.4f}")
print(f"Best Trial Loss: {best_result.metrics['best_loss']:.4f}")
print(f"\nExperiment saved to: {local_experiment_path}")
print(f"Drive backup: {experiment_path}")
print(f"To resume after Colab dies: just re-run this notebook.")


2026-04-03 16:09:19,824	INFO worker.py:2013 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
[I 2026-04-03 16:09:23,078] A new study created in memory with name: optuna


Dataset: /dev/shm/sunrgbd_19_traintest

STARTING NEW EXPERIMENT

STARTING HYPERPARAMETER TUNING
== Status ==
Current time: 2026-04-03 16:09:27 (running for 00:00:00.16)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 60.000: None | Iter 30.000: None | Iter 15.000: None
Logical resource usage: 0/12 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:A100)
Result logdir: /tmp/ray/session_2026-04-03_16-09-13_666604_3044/artifacts/2026-04-03_16-09-23/sun_rgbd_hpo_opt_mca_high_MD/driver_artifacts
Number of trials: 1/200 (1 PENDING)
+---------------------------+----------+-------+-------------+-------------+-------------+-------------+-------------------+------------------+----------------+---------------+------------------+-----------------+------------------------+
| Trial name                | status   | loc   |          lr |          wd |     eta_min |   dropout_p |   label_smoothing |   grad_clip_norm |   rgb_aug_prob |   rgb_aug_mag |   depth_aug_prob |   depth_aug_mag |   modality_dropout_rat 

(train_linet_tune pid=4743) /content/Multi-Stream-Neural-Networks/src/training/schedulers.py:193: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
(train_linet_tune pid=4743)   scheduler.step()


== Status ==
Current time: 2026-04-03 16:12:27 (running for 00:03:00.64)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 60.000: None | Iter 30.000: None | Iter 15.000: None
Logical resource usage: 10.0/12 CPUs, 0.9999999999999999/1 GPUs (0.0/1.0 accelerator_type:A100)
Result logdir: /tmp/ray/session_2026-04-03_16-09-13_666604_3044/artifacts/2026-04-03_16-09-23/sun_rgbd_hpo_opt_mca_high_MD/driver_artifacts
Number of trials: 10/200 (10 RUNNING)
+---------------------------+----------+------------------+-------------+-------------+-------------+-------------+-------------------+------------------+----------------+---------------+------------------+-----------------+------------------------+--------+----------------+------------------+-----------------+-------------+
| Trial name                | status   | loc              |          lr |          wd |     eta_min |   dropout_p |   label_smoothing |   grad_clip_norm |   rgb_aug_prob |   rgb_aug_mag |   depth_aug_prob |   depth_aug_mag 

(train_linet_tune pid=4834) /content/Multi-Stream-Neural-Networks/src/training/schedulers.py:193: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
(train_linet_tune pid=4834)   scheduler.step()


...

== Status ==
Current time: 2026-04-03 23:33:18 (running for 07:23:51.43)
Using AsyncHyperBand: num_stopped=169
Bracket: Iter 60.000: 0.5886127466433927 | Iter 30.000: 0.48703511569060776 | Iter 15.000: 0.3955723858977619
Logical resource usage: 7.0/12 CPUs, 0.7/1 GPUs (0.0/1.0 accelerator_type:A100)
Result logdir: /tmp/ray/session_2026-04-03_16-09-13_666604_3044/artifacts/2026-04-03_16-09-23/sun_rgbd_hpo_opt_mca_high_MD/driver_artifacts
Number of trials: 200/200 (7 RUNNING, 193 TERMINATED)
+---------------------------+------------+--------------------+-------------+-------------+-------------+-------------+-------------------+------------------+----------------+---------------+------------------+-----------------+------------------------+--------+----------------+------------------+-----------------+-------------+
| Trial name                | status     | loc                |          lr |          wd |     eta_min |   dropout_p |   label_smoothing |   grad_clip_norm |   rgb_aug_prob 

2026-04-03 23:43:35,090	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/content/ray_results/sun_rgbd_hpo_opt_mca_high_MD' in 0.0629s.


== Status ==
Current time: 2026-04-03 23:43:35 (running for 07:34:08.05)
Using AsyncHyperBand: num_stopped=172
Bracket: Iter 60.000: 0.5888116853801828 | Iter 30.000: 0.487012032615511 | Iter 15.000: 0.3955723858977619
Logical resource usage: 1.0/12 CPUs, 0.1/1 GPUs (0.0/1.0 accelerator_type:A100)
Result logdir: /tmp/ray/session_2026-04-03_16-09-13_666604_3044/artifacts/2026-04-03_16-09-23/sun_rgbd_hpo_opt_mca_high_MD/driver_artifacts
Number of trials: 200/200 (200 TERMINATED)
+---------------------------+------------+--------------------+-------------+-------------+-------------+-------------+-------------------+------------------+----------------+---------------+------------------+-----------------+------------------------+--------+----------------+------------------+-----------------+-------------+
| Trial name                | status     | loc                |          lr |          wd |     eta_min |   dropout_p |   label_smoothing |   grad_clip_norm |   rgb_aug_prob |   rgb_aug_m

2026-04-03 23:43:48,361	INFO tune.py:1041 -- Total run time: 27265.29 seconds (27247.98 seconds for the tuning loop).


[DriveSyncCallback] synced to Drive (experiment end)

TUNING COMPLETE
Best Trial Config: {'lr': 0.000160413607942147, 'wd': 8.542687072848881e-05, 'eta_min': 5.026316506769537e-07, 'dropout_p': 0.3112667840901657, 'label_smoothing': 0.13064415768104926, 'grad_clip_norm': 1.0064104068700201, 'rgb_aug_prob': 0.7768323877987129, 'rgb_aug_mag': 0.797730990885669, 'depth_aug_prob': 0.8344935223364396, 'depth_aug_mag': 0.9970208105130746, 'modality_dropout_rate': 0.31978385833160383}
Best Trial Val MCA: 0.6523
Best Trial Accuracy: 0.7348
Best Trial Loss: 1.4754

Experiment saved to: /content/ray_results/sun_rgbd_hpo_opt_mca_high_MD
Drive backup: /content/drive/MyDrive/ray_tune_experiments/sun_rgbd_hpo_opt_mca_high_MD
To resume after Colab dies: just re-run this notebook.


In [12]:
# =============================================================================
# SAVE RESULTS CSV (for offline analysis)
# =============================================================================
# Ray Tune already saved everything to DRIVE_STORAGE_PATH.
# This cell just exports a clean CSV for easy analysis.
# =============================================================================

import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

results_df = results.get_dataframe()

csv_dir = f"{DRIVE_STORAGE_PATH}/analysis"
Path(csv_dir).mkdir(parents=True, exist_ok=True)

csv_path = f"{csv_dir}/sun_hpo_results_{timestamp}.csv"
results_df.to_csv(csv_path, index=False)
print(f"Results CSV saved: {csv_path}")
print(f"  Trials: {len(results_df)}")

latest_path = f"{csv_dir}/sun_hpo_results_latest.csv"
results_df.to_csv(latest_path, index=False)
print(f"Latest copy: {latest_path}")


Results CSV saved: /content/drive/MyDrive/ray_tune_experiments/analysis/sun_hpo_results_20260403_234348.csv
  Trials: 200
Latest copy: /content/drive/MyDrive/ray_tune_experiments/analysis/sun_hpo_results_latest.csv


In [ ]:
# Analyze Top 10 Trials from Ray Tune (ranked by best_accuracy)
# With continuous search spaces, each trial has unique float values —
# no grouping by config. Treat fold assignment as noise.
import pandas as pd

print("=" * 80)
print("TOP 10 TRIALS BY BEST VAL MCA (CONTINUOUS SEARCH)")
print("=" * 80)

# Get all trials and convert to DataFrame
df = results.get_dataframe()

# Sort by best_accuracy (descending)
df_sorted = df.sort_values('best_val_mca', ascending=False)

# Select relevant columns for display
display_cols = [
    'best_val_mca', 'best_train_mca', 'best_accuracy', 'best_train_acc', 'composite',
    'config/lr',
    'config/wd',
    'config/eta_min',
    # 'config/t_max',
    'config/dropout_p',
    'config/label_smoothing',
    'config/grad_clip_norm',
    'config/rgb_aug_prob',
    'config/rgb_aug_mag',
    'config/depth_aug_prob',
    'config/depth_aug_mag',
    'config/stem_lr_multiplier',
    'config/modality_dropout_rate',
    # 'config/modality_dropout_start', 'config/modality_dropout_ramp',
]

# Get top 10 trials
top_10 = df_sorted[display_cols].head(10)

# Format for better display
top_10_formatted = top_10.copy()
top_10_formatted['best_val_mca'] = top_10_formatted['best_val_mca'].apply(lambda x: f"{x*100:.2f}%")
top_10_formatted['best_train_mca'] = top_10_formatted['best_train_mca'].apply(lambda x: f"{x*100:.2f}%")
top_10_formatted['best_accuracy'] = top_10_formatted['best_accuracy'].apply(lambda x: f"{x*100:.2f}%")

# Format scientific notation columns
sci_cols = [
    'config/lr', 'config/wd',
]
for col in sci_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.2e}")

# Format float columns
float_cols = [
    'config/dropout_p',
    'config/label_smoothing',
    'config/grad_clip_norm',
    'config/rgb_aug_prob',
    'config/rgb_aug_mag',
    'config/depth_aug_prob',
    'config/depth_aug_mag',
    'config/stem_lr_multiplier',
    'config/modality_dropout_rate',
]
for col in float_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.3f}")

print(top_10_formatted.to_string(index=False))
print("\n" + "=" * 80)


TOP 10 TRIALS BY BEST VAL MCA (CONTINUOUS SEARCH)
best_val_mca best_train_mca best_accuracy  best_train_acc  composite config/lr config/wd config/eta_min config/dropout_p config/label_smoothing config/grad_clip_norm config/rgb_aug_prob config/rgb_aug_mag config/depth_aug_prob config/depth_aug_mag config/modality_dropout_rate
      65.23%         89.25%        73.48%        0.911507   0.582954  1.60e-04  8.54e-05       5.03e-07            0.311                  0.131                 1.006               0.777              0.798                 0.834                0.997                        0.320
      64.47%         88.43%        73.48%        0.914861   0.575978  1.43e-04  1.65e-04       9.34e-07            0.281                  0.123                 1.018               0.827              0.724                 0.752                0.938                        0.278
      64.13%         89.50%        71.83%        0.894737   0.559622  1.45e-04  7.96e-05       5.63e-07            0.30

In [14]:
# =============================================================================
# ANALYZE TOP 10 TRIALS BY BEST VAL MCA
# =============================================================================

import pandas as pd

df = results.get_dataframe()
df = df.sort_values("best_val_mca", ascending=False)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]

print("=" * 80)
print("TOP 10 TRIALS BY BEST VAL MCA")
print("=" * 80)

for rank, (_, row) in enumerate(top_10.iterrows(), 1):
    gap = row.get("best_train_mca", 0) - row.get("best_val_mca", 0)
    print(f"\n--- #{rank} | Best Val MCA: {row['best_val_mca']*100:.2f}% | "
          f"Val MCA: {row['best_val_mca']*100:.2f}% | Acc: {row['best_accuracy']*100:.2f}% | "
          f"Gap: {gap*100:.1f}pp ---")

print("\n" + "=" * 80)
print("HYPERPARAMETER RANGES ACROSS TOP 10")
print("=" * 80)
print(f"{'Parameter':<35} {'Min':>12} {'Max':>12} {'Median':>12}")
print("-" * 75)

for col in config_cols:
    short_name = col.replace("config/", "")
    col_min = top_10[col].min()
    col_max = top_10[col].max()
    col_med = top_10[col].median()
    if abs(col_med) < 0.001:
        print(f"{short_name:<35} {col_min:>12.2e} {col_max:>12.2e} {col_med:>12.2e}")
    else:
        print(f"{short_name:<35} {col_min:>12.4f} {col_max:>12.4f} {col_med:>12.4f}")


TOP 10 TRIALS BY BEST VAL MCA

--- #1 | Best Val MCA: 65.23% | Val MCA: 65.23% | Acc: 73.48% | Gap: 24.0pp ---

--- #2 | Best Val MCA: 64.47% | Val MCA: 64.47% | Acc: 73.48% | Gap: 24.0pp ---

--- #3 | Best Val MCA: 64.13% | Val MCA: 64.13% | Acc: 71.83% | Gap: 25.4pp ---

--- #4 | Best Val MCA: 63.94% | Val MCA: 63.94% | Acc: 70.07% | Gap: 22.2pp ---

--- #5 | Best Val MCA: 63.76% | Val MCA: 63.76% | Acc: 72.45% | Gap: 25.1pp ---

--- #6 | Best Val MCA: 63.54% | Val MCA: 63.54% | Acc: 71.72% | Gap: 26.0pp ---

--- #7 | Best Val MCA: 63.39% | Val MCA: 63.39% | Acc: 72.14% | Gap: 30.0pp ---

--- #8 | Best Val MCA: 62.92% | Val MCA: 62.92% | Acc: 70.38% | Gap: 21.3pp ---

--- #9 | Best Val MCA: 62.85% | Val MCA: 62.85% | Acc: 70.49% | Gap: 20.9pp ---

--- #10 | Best Val MCA: 62.58% | Val MCA: 62.58% | Acc: 70.79% | Gap: 22.9pp ---

HYPERPARAMETER RANGES ACROSS TOP 10
Parameter                                    Min          Max       Median
-----------------------------------------------

In [15]:
# =============================================================================
# FULL CONFIG TABLE — TOP 10 BY BEST VAL MCA
# =============================================================================

import pandas as pd

df = results.get_dataframe()
df = df.sort_values("best_val_mca", ascending=False)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]
top_10["gap"] = top_10.get("best_train_mca", 0) - top_10.get("best_val_mca", 0)

display_df = top_10[["best_val_mca", "best_val_mca", "best_accuracy", "training_iteration"] + config_cols].copy()
display_df.insert(0, "rank", range(1, len(display_df) + 1))
display_df["best_val_mca"] = display_df["best_val_mca"].apply(lambda x: f"{x*100:.2f}%")
display_df["best_accuracy"] = display_df["best_accuracy"].apply(lambda x: f"{x*100:.2f}%")
display_df["training_iteration"] = display_df["training_iteration"].astype(int)

sci_cols = [c for c in config_cols if any(k in c for k in ["lr_", "wd_", "eta_min"]) and "multiplier" not in c]
float_cols = [c for c in config_cols if c not in sci_cols and c != "config/t_max"]

for col in sci_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.2e}")
for col in float_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(
            lambda x: f"{int(x)}" if col == "config/t_max" else f"{x:.3f}"
        )

display_df.columns = [c.replace("config/", "") for c in display_df.columns]

print(display_df.to_string(index=False))


TypeError: unsupported format string passed to Series.__format__